# Evaluate Qwen3-0.6B as a calculator agent

This notebook loads the 1,000-row aggregate dataset, selects its held-out test split, applies Qwen3's real chat template, executes model-requested calculator calls, and measures answer and trajectory accuracy. Each trajectory follows a strict one-call protocol: generate one tool call, execute it, append its result, then generate again. Active trajectories are dynamically batched with left padding and prompt-length bucketing to keep the GPU utilized.

Primary metrics:
- final-answer accuracy
- valid tool-use rate
- exact calculator-trace accuracy
- correct tool-call-count rate

Per-example predictions and aggregate metrics are written remotely to `/content/outputs/evaluations/` and preserved locally under `outputs/evaluations/`.

In [ ]:
# @title Install runtime dependencies
%pip install -q "transformers==5.9.0" accelerate tqdm

In [ ]:
# @title Configuration and dataset loading
from __future__ import annotations

import hashlib
import json
import re
from importlib.metadata import version
from pathlib import Path
from statistics import mean
from typing import Any

import torch
import transformers
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed

MODEL_ID = "Qwen/Qwen3-0.6B"
DATASET_PATH = Path("/content/data/calculator_qwen3")
SPLIT = "test"
MAX_EXAMPLES = None  # Set an integer for a quick smoke test.
SEED = 20260719
BATCH_SIZE = 16
MAX_TOOL_ROUNDS = 6
MAX_TOOL_CALLS = 12
MAX_NEW_TOKENS = 128
ENABLE_THINKING = False  # Set True for a slower thinking-mode benchmark.
RESUME = True
EVALUATOR_VERSION = 2
RESULTS_DIR = Path("/content/outputs/evaluations")

set_seed(SEED)

if not DATASET_PATH.exists():
    local_path = Path("data/calculator_qwen3")
    if local_path.exists():
        DATASET_PATH = local_path
    else:
        raise FileNotFoundError(f"Dataset not found: {DATASET_PATH}")

from datasets import load_from_disk

def remove_null_fields(value):
    if isinstance(value, dict):
        return {key: remove_null_fields(child) for key, child in value.items() if child is not None}
    if isinstance(value, list):
        return [remove_null_fields(child) for child in value]
    return value

hf_dataset = load_from_disk(str(DATASET_PATH))
dataset_fingerprints = {name: split._fingerprint for name, split in hf_dataset.items()}
DATASET_SHA256 = hashlib.sha256(
    json.dumps(dataset_fingerprints, sort_keys=True).encode("utf-8")
).hexdigest()
all_records_count = sum(len(split) for split in hf_dataset.values())
records = [remove_null_fields(dict(row)) for row in hf_dataset[SPLIT]]
if MAX_EXAMPLES is not None:
    records = records[:MAX_EXAMPLES]

assert all_records_count == 1_000, f"Expected 1,000 rows, found {all_records_count}"
assert records, f"No rows found for split={SPLIT!r}"
SELECTED_RECORDS_SHA256 = hashlib.sha256(
    json.dumps(records, sort_keys=True, separators=(",", ":")).encode("utf-8")
).hexdigest()
print(f"Loaded {all_records_count:,} rows; evaluating {len(records):,} {SPLIT} rows")
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
# @title Load Qwen3-0.6B for batched inference
DTYPE = torch.float16 if torch.cuda.is_available() else torch.float32

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.padding_side = "left"  # Required for decoder-only batched generation.
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=DTYPE,
    device_map="auto",
    attn_implementation="sdpa",
)
model.eval()
MODEL_REVISION = (
    getattr(model.config, "_commit_hash", None)
    or tokenizer.init_kwargs.get("_commit_hash")
    or "unresolved"
)
DEVICE_NAME = (
    torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
)
RUNTIME_METADATA = {
    "model_revision": MODEL_REVISION,
    "transformers_version": transformers.__version__,
    "accelerate_version": version("accelerate"),
    "torch_version": torch.__version__,
    "dtype": str(next(model.parameters()).dtype),
    "device": DEVICE_NAME,
    "seed": SEED,
    "dataset_sha256": DATASET_SHA256,
    "selected_records_sha256": SELECTED_RECORDS_SHA256,
    "evaluator_version": EVALUATOR_VERSION,
}
RUN_CONFIG = {
    "model": MODEL_ID,
    "model_revision": MODEL_REVISION,
    "dataset_sha256": DATASET_SHA256,
    "selected_records_sha256": SELECTED_RECORDS_SHA256,
    "split": SPLIT,
    "max_examples": MAX_EXAMPLES,
    "num_examples": len(records),
    "batch_size": BATCH_SIZE,
    "max_new_tokens": MAX_NEW_TOKENS,
    "max_tool_rounds": MAX_TOOL_ROUNDS,
    "max_tool_calls": MAX_TOOL_CALLS,
    "enable_thinking": ENABLE_THINKING,
    "do_sample": False,
    "use_cache": True,
    "stop_strings": ["</tool_call>"],
    "padding_side": tokenizer.padding_side,
    "transformers_version": transformers.__version__,
    "torch_version": torch.__version__,
    "dtype": str(next(model.parameters()).dtype),
    "device": DEVICE_NAME,
    "seed": SEED,
    "evaluator_version": EVALUATOR_VERSION,
}
RUN_SIGNATURE = hashlib.sha256(
    json.dumps(RUN_CONFIG, sort_keys=True).encode("utf-8")
).hexdigest()[:16]
RUNTIME_METADATA["run_signature"] = RUN_SIGNATURE
print(
    f"Loaded {MODEL_ID}@{MODEL_REVISION} with dtype={DTYPE}, "
    f"batch_size={BATCH_SIZE}, thinking={ENABLE_THINKING}"
)

In [ ]:
# @title Tool execution and response parsing
TOOL_TAG_RE = re.compile(r"<tool_call>\s*(.*?)\s*</tool_call>", re.DOTALL)
ANSWER_PATTERNS = (
    re.compile(
        r"(?:the\s+)?answer\s+is\s*[:=]?\s*\$?\s*(-?\d+)\s*\$?",
        re.IGNORECASE,
    ),
    re.compile(
        r"(?:the\s+)?(?:final\s+)?result(?:\s+of.*?)?\s+is\s*[:=]?\s*\$?\s*(-?\d+)\s*\$?",
        re.IGNORECASE | re.DOTALL,
    ),
    re.compile(r"\\boxed\{\s*(-?\d+)\s*\}"),
    re.compile(
        r"final(?:\s+answer)?\s*[:=]\s*\$?\s*(-?\d+)\s*\$?",
        re.IGNORECASE,
    ),
    re.compile(r"^\s*\$?\s*(-?\d+)\s*\$?\s*\.?\s*$"),
)


def calculator(op: str, a: int, b: int) -> int:
    if op == "+":
        return a + b
    if op == "-":
        return a - b
    if op == "*":
        return a * b
    raise ValueError(f"Unsupported calculator operation: {op}")


def validate_semantic_trace(
    expression: str,
    calls: list[dict[str, Any]],
    final_answer: int,
) -> tuple[bool, bool]:
    """Accept any legal precedence-respecting reduction order."""
    pieces = re.findall(r"\d+|[+*-]", expression)
    initial = tuple(int(piece) if piece.isdigit() else piece for piece in pieces)
    possible_states: set[tuple[int | str, ...]] = {initial}

    for call in calls:
        op, a, b = call["op"], call["a"], call["b"]
        next_states: set[tuple[int | str, ...]] = set()
        for tokens in possible_states:
            multiplication_remains = "*" in tokens
            for index in range(1, len(tokens), 2):
                if tokens[index] != op:
                    continue
                # Multiplications may be reduced in any independent order. Once
                # they are gone, addition/subtraction must remain left-associative.
                if op in {"+", "-"} and (multiplication_remains or index != 1):
                    continue
                if tokens[index - 1] != a or tokens[index + 1] != b:
                    continue
                result = calculator(op, a, b)
                reduced = tokens[: index - 1] + (result,) + tokens[index + 2 :]
                next_states.add(reduced)
        if not next_states:
            return False, False
        possible_states = next_states

    complete = any(
        len(tokens) == 1 and tokens[0] == final_answer
        for tokens in possible_states
    )
    return bool(calls), complete


def parse_tool_call(text: str) -> tuple[dict[str, Any] | None, str | None]:
    matches = TOOL_TAG_RE.findall(text)
    if not matches:
        if "<tool_call>" in text:
            return None, "unclosed_tool_call"
        return None, None
    if len(matches) != 1:
        return None, "multiple_tool_calls_in_one_turn"
    try:
        payload = json.loads(matches[0])
        function_name = payload["name"]
        arguments = payload["arguments"]
        if function_name != "calculator":
            return None, "wrong_tool_name"
        if set(arguments) != {"op", "a", "b"}:
            return None, "wrong_argument_schema"
        if arguments["op"] not in {"+", "-", "*"}:
            return None, "invalid_operator"
        if type(arguments["a"]) is not int or type(arguments["b"]) is not int:
            return None, "non_integer_operand"
        return {"name": function_name, "arguments": arguments}, None
    except (KeyError, TypeError, json.JSONDecodeError):
        return None, "malformed_tool_call"


def parse_final_answer(text: str) -> int | None:
    for pattern in ANSWER_PATTERNS:
        matches = pattern.findall(text)
        if matches:
            return int(matches[-1])
    return None


def render_prompt(messages: list[dict[str, Any]], tools: list[dict[str, Any]]) -> str:
    return tokenizer.apply_chat_template(
        messages,
        tools=tools,
        add_generation_prompt=True,
        tokenize=False,
        enable_thinking=ENABLE_THINKING,
    )


def generate_turns(prompts: list[str]) -> list[str]:
    """Left-pad and generate one agent turn for a batch of active trajectories."""
    inputs = tokenizer(
        prompts,
        padding=True,
        truncation=False,
        add_special_tokens=False,
        return_tensors="pt",
    )
    inputs = {key: value.to(model.device) for key, value in inputs.items()}
    padded_input_width = inputs["input_ids"].shape[1]
    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            use_cache=True,
            stop_strings=["</tool_call>"],
            tokenizer=tokenizer,
            pad_token_id=tokenizer.pad_token_id,
        )
    generated = output[:, padded_input_width:]
    return tokenizer.batch_decode(generated, skip_special_tokens=True)


def expected_trace(record: dict[str, Any]) -> list[dict[str, Any]]:
    return [
        message["tool_calls"][0]["function"]["arguments"]
        for message in record["messages"]
        if message["role"] == "assistant" and "tool_calls" in message
    ]

In [ ]:
# @title Batched agent state and per-example scoring
import time


def new_state(record: dict[str, Any]) -> dict[str, Any]:
    return {
        "record": record,
        "messages": [dict(message) for message in record["messages"][:2]],
        "tools": record["tools"],
        "expected_calls": expected_trace(record),
        "model_calls": [],
        "generated_turns": [],
        "predicted_answer": None,
        "error": None,
        "tool_calls_valid": True,
        "done": False,
        "started": time.perf_counter(),
    }


def process_turn(state: dict[str, Any], text: str, turn_index: int) -> None:
    text = text.strip()
    # Enforce one tool call -> result -> next generation.
    closing_tag = "</tool_call>"
    first_call_end = text.find(closing_tag)
    if first_call_end >= 0:
        text = text[: first_call_end + len(closing_tag)]

    state["generated_turns"].append(text)
    parsed_call, parse_error = parse_tool_call(text)

    if parse_error is not None:
        state["error"] = parse_error
        state["tool_calls_valid"] = False
        state["done"] = True
        return
    if parsed_call is None:
        state["predicted_answer"] = parse_final_answer(text)
        if state["predicted_answer"] is None:
            state["error"] = "missing_final_answer"
        state["done"] = True
        return
    if len(state["model_calls"]) >= MAX_TOOL_CALLS:
        state["error"] = "max_tool_calls_exceeded"
        state["done"] = True
        return

    arguments = parsed_call["arguments"]
    try:
        result = calculator(arguments["op"], arguments["a"], arguments["b"])
    except ValueError:
        state["error"] = "calculator_rejected_call"
        state["tool_calls_valid"] = False
        state["done"] = True
        return

    call_id = f"eval_call_{turn_index:02d}"
    state["messages"].append(
        {
            "role": "assistant",
            "content": text.split("<tool_call>", 1)[0].strip(),
            "tool_calls": [
                {
                    "id": call_id,
                    "type": "function",
                    "function": {"name": "calculator", "arguments": arguments},
                }
            ],
        }
    )
    state["messages"].append(
        {
            "role": "tool",
            "name": "calculator",
            "tool_call_id": call_id,
            "content": json.dumps({"result": result}),
        }
    )
    state["model_calls"].append(arguments)


def finalize_state(state: dict[str, Any]) -> dict[str, Any]:
    record = state["record"]
    final_answer = record["metadata"]["final_answer"]
    predicted_answer = state["predicted_answer"]
    model_calls = state["model_calls"]
    expected_calls = state["expected_calls"]
    answer_correct = predicted_answer == final_answer
    exact_trace = model_calls == expected_calls
    valid_tool_use = bool(model_calls) and state["tool_calls_valid"]
    parseable_completion = predicted_answer is not None
    semantic_trace_valid, semantic_trace_complete = validate_semantic_trace(
        record["expression"], model_calls, final_answer
    )
    task_success = (
        answer_correct
        and valid_tool_use
        and semantic_trace_complete
    )
    return {
        "id": record["id"],
        "run_signature": RUN_SIGNATURE,
        "expression": record["expression"],
        "tier": record["metadata"]["tier"],
        "final_answer": final_answer,
        "predicted_answer": predicted_answer,
        "answer_correct": answer_correct,
        "used_tool": bool(model_calls),
        "valid_tool_use": valid_tool_use,
        "parseable_completion": parseable_completion,
        "tool_call_count": len(model_calls),
        "expected_tool_call_count": len(expected_calls),
        "tool_call_count_correct": len(model_calls) == len(expected_calls),
        "semantic_trace_valid": semantic_trace_valid,
        "semantic_trace_complete": semantic_trace_complete,
        "exact_trace": exact_trace,
        "task_success": task_success,
        "reference_trace_success": task_success and exact_trace,
        "error": state["error"],
        "model_calls": model_calls,
        "expected_calls": expected_calls,
        "generated_turns": state["generated_turns"],
        "completion_time_from_start_seconds": round(
            time.perf_counter() - state["started"], 3
        ),
    }

In [ ]:
# @title Run dynamically batched held-out evaluation
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
predictions_path = RESULTS_DIR / "predictions.jsonl"
target_ids = {record["id"] for record in records}
existing_results: list[dict[str, Any]] = []

if RESUME and predictions_path.exists():
    candidates: list[dict[str, Any]] = []
    with predictions_path.open(encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, start=1):
            if not line.strip():
                continue
            try:
                row = json.loads(line)
            except json.JSONDecodeError:
                print(f"Ignoring malformed checkpoint line {line_number}")
                continue
            if row.get("id") in target_ids:
                candidates.append(row)
    if candidates and all(
        row.get("run_signature") == RUN_SIGNATURE for row in candidates
    ):
        existing_by_id = {row["id"]: row for row in candidates}
        existing_results = list(existing_by_id.values())
        print(f"Resuming from {len(existing_results)} completed examples")
    elif candidates:
        print("Ignoring checkpoint from a different evaluation configuration")

completed_ids = {row["id"] for row in existing_results}
states = [new_state(record) for record in records if record["id"] not in completed_ids]
results: list[dict[str, Any]] = list(existing_results)
evaluation_started = time.perf_counter()

# Rewrite the checkpoint with only compatible rows before appending new results.
with predictions_path.open("w", encoding="utf-8") as handle, tqdm(
    total=len(records),
    initial=len(existing_results),
    desc=f"Evaluating {MODEL_ID}",
) as progress:
    for result in sorted(existing_results, key=lambda row: row["id"]):
        handle.write(json.dumps(result, ensure_ascii=False) + "\n")
    handle.flush()

    for turn_index in range(MAX_TOOL_ROUNDS + 1):
        active_states = [state for state in states if not state["done"]]
        if not active_states:
            break

        rendered_prompts = [
            render_prompt(state["messages"], state["tools"])
            for state in active_states
        ]
        tokenized_for_lengths = tokenizer(
            rendered_prompts, add_special_tokens=False
        )["input_ids"]
        prompt_states = [
            (len(token_ids), prompt, state)
            for token_ids, prompt, state in zip(
                tokenized_for_lengths, rendered_prompts, active_states
            )
        ]
        prompt_states.sort(key=lambda item: item[0])

        for start in range(0, len(prompt_states), BATCH_SIZE):
            batch = prompt_states[start : start + BATCH_SIZE]
            prompts = [prompt for _, prompt, _ in batch]
            batch_states = [state for _, _, state in batch]
            responses = generate_turns(prompts)

            for state, response in zip(batch_states, responses):
                process_turn(state, response, turn_index)
                if state["done"]:
                    result = finalize_state(state)
                    results.append(result)
                    handle.write(json.dumps(result, ensure_ascii=False) + "\n")
                    handle.flush()
                    progress.update(1)

    unfinished = [state for state in states if not state["done"]]
    for state in unfinished:
        state["error"] = "agent_loop_exhausted"
        state["done"] = True
        result = finalize_state(state)
        results.append(result)
        handle.write(json.dumps(result, ensure_ascii=False) + "\n")
        handle.flush()
        progress.update(1)

wall_time_seconds = time.perf_counter() - evaluation_started
new_examples_evaluated = len(results) - len(existing_results)
examples_per_second = (
    new_examples_evaluated / wall_time_seconds
    if new_examples_evaluated
    else 0.0
)
results.sort(key=lambda row: row["id"])
print(
    f"Saved {len(results):,} predictions to {predictions_path}; "
    f"evaluated {new_examples_evaluated:,} new examples in {wall_time_seconds:.1f}s "
    f"({examples_per_second:.2f} examples/s)"
)

In [ ]:
# @title Aggregate and save metrics
from collections import Counter

METRIC_FIELDS = (
    "answer_correct",
    "used_tool",
    "valid_tool_use",
    "parseable_completion",
    "tool_call_count_correct",
    "semantic_trace_valid",
    "semantic_trace_complete",
    "exact_trace",
    "task_success",
    "reference_trace_success",
)


def rates(rows: list[dict[str, Any]]) -> dict[str, float]:
    return {
        name: round(mean(float(row[name]) for row in rows), 4)
        for name in METRIC_FIELDS
    }


metrics: dict[str, Any] = {
    "model": MODEL_ID,
    "dataset": str(DATASET_PATH),
    "split": SPLIT,
    "num_examples": len(results),
    "reproducibility": RUNTIME_METADATA,
    "inference_config": {
        "batch_size": BATCH_SIZE,
        "max_new_tokens": MAX_NEW_TOKENS,
        "max_tool_rounds": MAX_TOOL_ROUNDS,
        "max_tool_calls": MAX_TOOL_CALLS,
        "do_sample": False,
        "use_cache": True,
        "stop_strings": ["</tool_call>"],
        "enable_thinking": ENABLE_THINKING,
        "padding_side": tokenizer.padding_side,
        "prompt_bucketing": "token_count",
        "tool_protocol": "one_call_then_result",
    },
    "metric_definitions": {
        "answer_correct": "Parsed final number equals the reference answer.",
        "valid_tool_use": "At least one tool call was made and every parsed call was schema-valid.",
        "parseable_completion": "A supported final-answer form was parsed.",
        "semantic_trace_valid": "Every call is a legal precedence-respecting expression reduction.",
        "semantic_trace_complete": "The calls legally reduce the full expression to the reference answer.",
        "exact_trace": "Model calls exactly match the single stored reference trace; alternative valid orders may score false.",
        "task_success": "Answer is correct, calls are schema-valid, and the semantic trace fully solves the expression.",
        "reference_trace_success": "Task success plus exact reference-trace match.",
    },
    "overall": rates(results),
    "by_tier": {},
    "errors": dict(Counter(row["error"] or "none" for row in results)),
    "resumed_examples": len(existing_results),
    "new_examples_evaluated": new_examples_evaluated,
    "wall_time_seconds": round(wall_time_seconds, 3),
    "examples_per_second": round(examples_per_second, 4),
}
for tier in ("easy", "medium", "hard"):
    tier_rows = [row for row in results if row["tier"] == tier]
    if tier_rows:
        metrics["by_tier"][tier] = {
            "num_examples": len(tier_rows),
            **rates(tier_rows),
        }

metrics_path = RESULTS_DIR / "metrics.json"
metrics_path.write_text(json.dumps(metrics, indent=2), encoding="utf-8")
print(json.dumps(metrics, indent=2))
print(f"Saved metrics to {metrics_path}")

In [ ]:
# @title Inspect representative task failures
failures = [row for row in results if not row["task_success"]]
print(f"Task failures: {len(failures)}/{len(results)}")
for row in failures[:5]:
    print("\n", "=" * 80)
    print("ID:", row["id"], "Tier:", row["tier"])
    print("Expression:", row["expression"])
    print("Expected answer:", row["final_answer"], "Predicted:", row["predicted_answer"])
    print("Error:", row["error"])
    print("Expected calls:", row["expected_calls"])
    print("Model calls:", row["model_calls"])
    print("Last turn:", row["generated_turns"][-1] if row["generated_turns"] else "<none>")

## Result interpretation

Use `task_success` for policy-aware correctness. It requires the right answer and a complete, precedence-respecting semantic reduction, while allowing mathematically valid operation orders that differ from the stored reference. `exact_trace` only measures equality with that single reference trace. `valid_tool_use` measures tool-call schema validity independently from whether the model produced a parseable final response.

The evaluator enforces the training protocol at generation time: stop after one complete tool call, execute it, append its result, and only then generate the next turn. The printed metrics above are the source of truth for this run.

## Example generation trace

Example `calculator_0884` demonstrates the strict wait cycle.

**Problem:** `14 - 18*7 + 18*5 + 18`  
**Reference answer:** `-4`

**Generation 1**

```text
<tool_call>
{"name": "calculator", "arguments": {"op": "-", "a": 14, "b": 18}}
</tool_call>
```

The evaluator stops at the first closing tool tag, executes only that call, and returns:

```json
{"result": -4}
```

**Generation 2 (after receiving the tool result)**

```text
The result is -4.
```

The final number happens to match the reference answer, but the model did not reduce the full expression. The expected calls were `18*7`, `18*5`, `14-126`, `-112+90`, and `-22+18`. Therefore this row has `answer_correct=true`, but `semantic_trace_complete=false`, `task_success=false`, and `exact_trace=false`.